# 01 -- Cryptographie avancee avec Python

## Objectifs pedagogiques

A la fin de ce notebook, vous saurez :

- utiliser `hashlib` pour les fonctions de hachage (SHA-256, SHA-3, BLAKE2) ;
- utiliser `hmac` pour les codes d'authentification ;
- generer des secrets cryptographiques avec le module `secrets` ;
- hacher des mots de passe avec `argon2-cffi` ;
- chiffrer/dechiffrer avec Fernet (symetrique) ;
- comprendre les bases du chiffrement asymetrique (RSA, Ed25519) ;
- signer et verifier des donnees ;
- gerer les certificats X.509 basiques.

## Prerequis -- ce que vous devez maitriser

- les fonctions Python, bytes vs str ;
- les notions de base de securite (chiffrement, hachage, signature) ;

Ce que ce notebook **ne couvrira pas** :

- la cryptographie mathematique en profondeur ;
- TLS/SSL en detail (mais on mentionne les certificats) ;
- les HSM et la gestion de cles en production.

## Plan

1. `hashlib` -- fonctions de hachage
2. `hmac` -- authentification de messages
3. `secrets` -- generation securisee
4. Hachage de mots de passe (`argon2-cffi`)
5. Chiffrement symetrique (`cryptography.fernet`)
6. Chiffrement symetrique avance (AES-GCM)
7. Chiffrement asymetrique (RSA)
8. Signatures (Ed25519)
9. Certificats X.509 basiques
10. Bonnes pratiques
11. Synthese
12. Exercices

---

## 1. `hashlib` -- fonctions de hachage

Une fonction de hachage transforme un message de taille arbitraire en une empreinte de taille fixe.

In [ ]:
import hashlib

hashlib.sha256(b'hello').hexdigest()

In [ ]:
hashlib.sha3_256(b'hello').hexdigest()

In [ ]:
hashlib.blake2b(b'hello', digest_size=32).hexdigest()

**BLAKE2** est plus rapide que SHA-256 et aussi sur. Preferer `sha256` pour la compatibilite, `blake2b` pour la performance.

### Hachage par blocs (fichiers volumineux)

In [ ]:
def hash_file(path: str, algo: str = 'sha256') -> str:
    h = hashlib.new(algo)
    with open(path, 'rb') as f:
        while chunk := f.read(8192):
            h.update(chunk)
    return h.hexdigest()

# hash_file('/etc/hostname')

### Algorithmes disponibles

In [ ]:
sorted(hashlib.algorithms_available)[:10]

---

## 2. `hmac` -- authentification de messages

Un HMAC (Hash-based Message Authentication Code) combine un hash avec une cle secrete.

In [ ]:
import hmac

key = b'ma_cle_secrete'
message = b'donnees importantes'

h = hmac.new(key, message, hashlib.sha256)
h.hexdigest()

### Comparaison en temps constant

In [ ]:
signature1 = hmac.new(key, message, hashlib.sha256).hexdigest()
signature2 = hmac.new(key, message, hashlib.sha256).hexdigest()

# TOUJOURS utiliser compare_digest, JAMAIS ==
hmac.compare_digest(signature1, signature2)

`compare_digest` empeche les timing attacks : la comparaison prend toujours le meme temps, que les chaines matchent ou non.

---

## 3. `secrets` -- generation securisee

Le module `secrets` genere des tokens et valeurs aleatoires **cryptographiquement surs**.

In [ ]:
import secrets

secrets.token_hex(32)

In [ ]:
secrets.token_urlsafe(32)

In [ ]:
secrets.token_bytes(32)[:10]

In [ ]:
# Generer un mot de passe aleatoire
import string
alphabet = string.ascii_letters + string.digits + '!@#$%'
mot_de_passe = ''.join(secrets.choice(alphabet) for _ in range(16))
mot_de_passe

**Ne jamais utiliser `random`** pour la cryptographie. `random` est previsible. `secrets` utilise `os.urandom` en interne.

---

## 4. Hachage de mots de passe (`argon2-cffi`)

**Ne jamais stocker** un mot de passe en clair ou avec un simple SHA-256. Utiliser un algorithme **lent et sale** (Argon2, bcrypt, scrypt).

In [ ]:
# pip install argon2-cffi
from argon2 import PasswordHasher

ph = PasswordHasher()
hash_str = ph.hash('mon_mot_de_passe_secret')
hash_str

In [ ]:
# Verification
ph.verify(hash_str, 'mon_mot_de_passe_secret')

In [ ]:
# Mauvais mot de passe
try:
    ph.verify(hash_str, 'mauvais')
except Exception as e:
    print(f'Erreur : {type(e).__name__}')

Argon2 est le gagnant du Password Hashing Competition (2015). C'est le standard recommande.

---

## 5. Chiffrement symetrique (`cryptography.fernet`)

Fernet est un schema de chiffrement symetrique simple et sur (AES-128-CBC + HMAC-SHA256).

In [ ]:
from cryptography.fernet import Fernet

key = Fernet.generate_key()
f = Fernet(key)

message = b'donnees sensibles'
token = f.encrypt(message)
token[:50]

In [ ]:
f.decrypt(token)

La cle doit etre stockee de facon securisee (variable d'environnement, gestionnaire de secrets, KMS).

### Fernet avec TTL

In [ ]:
import time
token = f.encrypt(b'expire bientot')
time.sleep(0.1)
f.decrypt(token, ttl=10)  # OK si moins de 10 secondes

---

## 6. Chiffrement symetrique avance (AES-GCM)

Pour plus de controle, utiliser AES-GCM directement.

In [ ]:
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import os

key = AESGCM.generate_key(bit_length=256)
nonce = os.urandom(12)
aad = b'donnees associees non chiffrees'

aesgcm = AESGCM(key)
ct = aesgcm.encrypt(nonce, b'message secret', aad)
aesgcm.decrypt(nonce, ct, aad)

AES-GCM fournit le chiffrement **et** l'authentification en une seule operation (AEAD).

---

## 7. Chiffrement asymetrique (RSA)

Le chiffrement asymetrique utilise une paire cle publique / cle privee.

In [ ]:
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes

private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
public_key = private_key.public_key()
print(f'Cle generee : {private_key.key_size} bits')

### Chiffrer avec la cle publique

In [ ]:
ciphertext = public_key.encrypt(
    b'message secret',
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None,
    ),
)
ciphertext[:20]

### Dechiffrer avec la cle privee

In [ ]:
plaintext = private_key.decrypt(
    ciphertext,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None,
    ),
)
plaintext

---

## 8. Signatures (Ed25519)

Ed25519 est un algorithme de signature moderne, rapide et sur.

In [ ]:
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey

private_key = Ed25519PrivateKey.generate()
public_key = private_key.public_key()

message = b'document a signer'
signature = private_key.sign(message)
print(f'Signature : {signature.hex()[:40]}...')

In [ ]:
# Verification
public_key.verify(signature, message)
print('Signature valide')

In [ ]:
# Signature invalide
try:
    public_key.verify(signature, b'document modifie')
except Exception as e:
    print(f'Erreur : {type(e).__name__}')

---

## 9. Certificats X.509 basiques

In [ ]:
from cryptography import x509
from cryptography.x509.oid import NameOID
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import rsa
import datetime

# Generer un certificat auto-signe
key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
subject = issuer = x509.Name([
    x509.NameAttribute(NameOID.COUNTRY_NAME, 'FR'),
    x509.NameAttribute(NameOID.ORGANIZATION_NAME, 'Formation Python'),
    x509.NameAttribute(NameOID.COMMON_NAME, 'localhost'),
])
cert = (
    x509.CertificateBuilder()
    .subject_name(subject)
    .issuer_name(issuer)
    .public_key(key.public_key())
    .serial_number(x509.random_serial_number())
    .not_valid_before(datetime.datetime.now(datetime.UTC))
    .not_valid_after(datetime.datetime.now(datetime.UTC) + datetime.timedelta(days=365))
    .sign(key, hashes.SHA256())
)
print(f'CN = {cert.subject.get_attributes_for_oid(NameOID.COMMON_NAME)[0].value}')

---

## 10. Bonnes pratiques

1. **Ne jamais inventer** son propre algorithme cryptographique.
2. **Utiliser des bibliotheques maintenues** (`cryptography`, `argon2-cffi`).
3. **Ne jamais stocker** les cles en dur dans le code.
4. **Utiliser `secrets`** pour la generation aleatoire, pas `random`.
5. **Comparer les hashes** avec `hmac.compare_digest`, pas `==`.
6. **Argon2** pour les mots de passe, pas SHA-256.
7. **AES-GCM** ou **Fernet** pour le chiffrement symetrique.
8. **Ed25519** pour les signatures (preferable a RSA pour les nouveaux projets).

---

## Synthese

| Besoin | Outil |
|---|---|
| Empreinte de fichier | `hashlib.sha256` ou `blake2b` |
| Authentification de message | `hmac` + `compare_digest` |
| Token aleatoire | `secrets.token_hex` / `token_urlsafe` |
| Mot de passe | `argon2-cffi` |
| Chiffrement symetrique simple | `cryptography.fernet.Fernet` |
| Chiffrement symetrique avance | `AESGCM` |
| Chiffrement asymetrique | RSA + OAEP |
| Signature | Ed25519 |
| Certificat | `cryptography.x509` |

---

## Exercices

### Exercice 1 -- Verifier l'integrite d'un fichier

Ecrivez une fonction qui prend un chemin de fichier et un hash attendu (SHA-256 hex), et retourne True si le fichier correspond.

In [ ]:
def verify_file_integrity(path: str, expected_hash: str) -> bool:
    ...

<details>
<summary>Voir la correction</summary>

```python
import hashlib

def verify_file_integrity(path: str, expected_hash: str) -> bool:
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(8192):
            h.update(chunk)
    return h.hexdigest() == expected_hash
```

</details>

### Exercice 2 -- Gestionnaire de mots de passe simple

Creez une classe `PasswordStore` qui stocke des mots de passe haches avec Argon2. Methodes : `add(username, password)`, `verify(username, password) -> bool`.

In [ ]:
class PasswordStore:
    ...

<details>
<summary>Voir la correction</summary>

```python
from argon2 import PasswordHasher
from argon2.exceptions import VerifyMismatchError

class PasswordStore:
    def __init__(self):
        self._store: dict[str, str] = {}
        self._ph = PasswordHasher()

    def add(self, username: str, password: str) -> None:
        self._store[username] = self._ph.hash(password)

    def verify(self, username: str, password: str) -> bool:
        if username not in self._store:
            return False
        try:
            return self._ph.verify(self._store[username], password)
        except VerifyMismatchError:
            return False

store = PasswordStore()
store.add('alice', 'secret123')
print(store.verify('alice', 'secret123'))
print(store.verify('alice', 'wrong'))
```

</details>

### Exercice 3 -- Chiffrement de fichier

Ecrivez deux fonctions : `encrypt_file(path, key)` et `decrypt_file(path, key)` utilisant Fernet.

In [ ]:
from cryptography.fernet import Fernet
from pathlib import Path

def encrypt_file(path: Path, key: bytes) -> Path:
    ...

def decrypt_file(path: Path, key: bytes) -> Path:
    ...

<details>
<summary>Voir la correction</summary>

```python
from cryptography.fernet import Fernet
from pathlib import Path

def encrypt_file(path: Path, key: bytes) -> Path:
    f = Fernet(key)
    data = Path(path).read_bytes()
    encrypted = f.encrypt(data)
    enc_path = Path(str(path) + '.enc')
    enc_path.write_bytes(encrypted)
    return enc_path

def decrypt_file(path: Path, key: bytes) -> Path:
    f = Fernet(key)
    data = Path(path).read_bytes()
    decrypted = f.decrypt(data)
    dec_path = Path(str(path).removesuffix('.enc'))
    dec_path.write_bytes(decrypted)
    return dec_path

key = Fernet.generate_key()
Path('/tmp/test_crypto.txt').write_text('secret')
enc = encrypt_file(Path('/tmp/test_crypto.txt'), key)
dec = decrypt_file(enc, key)
print(dec.read_text())
```

</details>

---

## Ressources externes

- [Documentation cryptography](https://cryptography.io/en/latest/)
- [Documentation hashlib](https://docs.python.org/3/library/hashlib.html)
- [Documentation secrets](https://docs.python.org/3/library/secrets.html)
- [argon2-cffi](https://argon2-cffi.readthedocs.io/)
- [Serious Cryptography (Aumasson)](https://nostarch.com/seriouscrypto)